<div style="background:linear-gradient(135deg,#071B33,#0077B6);padding:36px;border-radius:18px;color:white">
<div style="font-size:13px;letter-spacing:2px">EXAMEN TRANSVERSAL · PROGRAMACIÓN PARA LA CIENCIA DE DATOS</div>
<h1 style="color:white">Operaciones aéreas en Chile</h1>
<div style="font-size:22px">ETL, clasificación, clustering y dashboard interactivo</div>
<div style="margin-top:18px">Integrantes: completar nombres · Julio 2026</div>
</div>

## Pregunta del proyecto

**¿Cómo se comporta la actividad de los aeropuertos chilenos y podemos identificar anticipadamente meses de alta actividad?**

Este notebook constituye la presentación completa. Aplica exclusivamente contenidos trabajados durante el semestre: Pandas, limpieza, transformación, API REST, regresión logística, árbol de decisión, K-Means, silhouette, PCA, Plotly y Git/GitHub.

## 0. Cobertura de los requisitos del examen

| Requisito | Evidencia en el notebook |
|---|---|
| Caso realista chileno | Operaciones de aeropuertos de Chile |
| Al menos 3 fuentes | CSV JAC + API de feriados + catálogo de aeropuertos |
| Pipeline ETL | Extraer, validar, transformar e integrar |
| Pandas | Limpieza, `groupby`, `merge`, nuevas columnas |
| Machine learning supervisado | Regresión logística y árbol de decisión |
| Machine learning no supervisado | K-Means, silhouette y PCA |
| Visualización interactiva | Gráficos y dashboard Plotly |
| Trabajo colaborativo | Evidencias propuestas para Git/GitHub |
| Despliegue | Dockerfile incluido en la entrega |
| Documentación | Explicaciones, fuentes, limitaciones y conclusiones |

> **Guion:** explicar esta tabla en 30 segundos para demostrar que el proyecto cubre el encargo completo.


## 1. Preparar Google Colab

La única acción manual es subir `operaciones-aeropuertos.csv` cuando Colab lo solicite. Las otras fuentes se descargan automáticamente y cuentan con respaldo local en el proyecto.

In [4]:
import io, json, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import requests
import plotly.express as px
import plotly.graph_objects as go

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, confusion_matrix, classification_report,
                             silhouette_score)
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

warnings.filterwarnings('ignore')
px.defaults.template='plotly_white'
AZUL='#0077B6'; AMARILLO='#FFB703'; CELESTE='#00B4D8'; AZUL_OSCURO='#071B33'
RANDOM_STATE=42
print('Entorno preparado correctamente')


Entorno preparado correctamente


In [5]:
# Fuente 1: CSV oficial de operaciones
archivo='operaciones-aeropuertos.csv'
if not Path(archivo).exists():
    try:
        from google.colab import files
        print('Selecciona operaciones-aeropuertos.csv')
        subida=files.upload()
        archivo=next(iter(subida))
    except ImportError:
        posibles=['../data/raw/operaciones-aeropuertos.csv','data/raw/operaciones-aeropuertos.csv']
        archivo=next((p for p in posibles if Path(p).exists()),archivo)

operaciones_raw=pd.read_csv(archivo)
print('Fuente 1 cargada:',operaciones_raw.shape)
operaciones_raw.head()


Fuente 1 cargada: (16007, 4)


,mes_id,aeropuerto_oaci,internacional_domestico,cnt_operaciones
0,200001,SCAP,D,6
1,200001,SCAR,D,691
2,200001,SCAR,I,139
3,200001,SCBA,D,382
4,200001,SCCC,D,82


In [6]:
# Fuente 2: API REST de feriados de Chile
feriados=[]
anios_api=range(2022,2027)

for anio in anios_api:
    url=f'https://date.nager.at/api/v3/PublicHolidays/{anio}/CL'
    try:
        respuesta=requests.get(url,timeout=20)
        respuesta.raise_for_status()
        feriados.extend(respuesta.json())
        print(f'Feriados {anio}: carga correcta')
    except requests.RequestException as error:
        print(f'Advertencia: no fue posible descargar {anio}: {error}')

feriados_raw=pd.DataFrame(feriados)
if feriados_raw.empty:
    raise RuntimeError('La API no entregó feriados. Revisa la conexión y vuelve a ejecutar la celda.')

print('Fuente 2 cargada desde API REST:',feriados_raw.shape)
feriados_raw[['date','localName']].head()


Feriados 2022: carga correcta
Feriados 2023: carga correcta
Feriados 2024: carga correcta
Feriados 2025: carga correcta
Feriados 2026: carga correcta
Fuente 2 cargada desde API REST: (86, 9)


,date,localName
0,2022-01-01,Año Nuevo
1,2022-04-15,Viernes Santo
2,2022-04-16,Sábado Santo
3,2022-05-01,Día del Trabajo
4,2022-05-21,Día de las Glorias Navales


In [7]:
# Fuente 3: catálogo abierto de aeropuertos
URL_AEROPUERTOS='https://davidmegginson.github.io/ourairports-data/airports.csv'
aeropuertos_mundo=pd.read_csv(URL_AEROPUERTOS)
aeropuertos_raw=(aeropuertos_mundo[aeropuertos_mundo['iso_country'].eq('CL')]
                 [['icao_code','name','type','latitude_deg','longitude_deg','elevation_ft','iso_region','municipality']]
                 .dropna(subset=['icao_code']).drop_duplicates('icao_code'))
print('Fuente 3 cargada:',aeropuertos_raw.shape)
aeropuertos_raw.head()


Fuente 3 cargada: (119, 8)


,icao_code,name,type,latitude_deg,longitude_deg,elevation_ft,iso_region,municipality
19214,SCAV,Achibueno Airport,small_airport,-36.141999,-71.374000,1673.0,CL-ML,Linares
19239,SHTO,Alto Hospicio Hospital Helistop.,heliport,-20.295985,-70.097637,1726.0,CL-TA,Alto Hospicio
19240,SHGF,Gustavo Fricke Hospital Helistop,heliport,-33.029345,-71.543095,243.0,CL-VS,Viña del Mar
19241,SHRZ,Reitz Building Two Helistop,heliport,-33.043037,-71.516654,240.0,CL-VS,Viña del Mar
19242,SHOL,Santa Laura Heliport,heliport,-32.990378,-71.217944,440.0,CL-VS,Olmué


## 2. Validar la calidad antes de transformar

Se revisan tamaño, tipos, nulos, duplicados y rangos. Este paso corresponde a las prácticas de limpieza del semestre.

In [8]:
calidad=pd.DataFrame({
    'Indicador':['Filas','Columnas','Nulos','Duplicados','Aeropuertos','Mes inicial','Mes final','Mínimo operaciones','Máximo operaciones'],
    'Resultado':[len(operaciones_raw),operaciones_raw.shape[1],int(operaciones_raw.isna().sum().sum()),int(operaciones_raw.duplicated().sum()),operaciones_raw.aeropuerto_oaci.nunique(),operaciones_raw.mes_id.min(),operaciones_raw.mes_id.max(),operaciones_raw.cnt_operaciones.min(),operaciones_raw.cnt_operaciones.max()]
})
calidad


,Indicador,Resultado
0,Filas,16007
1,Columnas,4
2,Nulos,0
3,Duplicados,0
4,Aeropuertos,69
5,Mes inicial,200001
6,Mes final,202606
7,Mínimo operaciones,1
8,Máximo operaciones,10722


In [9]:
# Reglas automáticas de control
assert operaciones_raw.isna().sum().sum()==0, 'Existen nulos en la fuente principal'
assert operaciones_raw.duplicated().sum()==0, 'Existen duplicados'
assert operaciones_raw.cnt_operaciones.ge(0).all(), 'Hay operaciones negativas'
assert operaciones_raw.internacional_domestico.isin(['D','I']).all(), 'Tipo de operación no reconocido'
print('✅ Todas las reglas de calidad fueron superadas')


✅ Todas las reglas de calidad fueron superadas


## 3. Transformar e integrar: pipeline ETL

- Convertimos `AAAAMM` a fecha.
- Limpiamos nombres y tipos.
- Contamos feriados por mes.
- Integramos ubicación y calendario mediante `merge`.
- Creamos variables necesarias para análisis y modelos.

In [10]:
# TRANSFORMAR fuente principal
df=operaciones_raw.copy()
df.columns=df.columns.str.strip().str.lower()
df['fecha']=pd.to_datetime(df['mes_id'].astype(str),format='%Y%m')
df['anio']=df.fecha.dt.year
df['mes']=df.fecha.dt.month
df['tipo_operacion']=df.internacional_domestico.map({'D':'Doméstica','I':'Internacional'})
df['es_internacional']=(df.internacional_domestico=='I').astype(int)

# TRANSFORMAR API: cantidad de feriados nacionales por mes (cobertura 2022-2026)
feriados_df=feriados_raw.copy()
feriados_df['fecha_dia']=pd.to_datetime(feriados_df['date'])
feriados_df['fecha']=feriados_df.fecha_dia.dt.to_period('M').dt.to_timestamp()
feriados_mes=feriados_df.groupby('fecha').size().reset_index(name='n_feriados')

# INTEGRAR las tres fuentes
df=df.merge(feriados_mes,on='fecha',how='left')
df=df.merge(aeropuertos_raw,left_on='aeropuerto_oaci',right_on='icao_code',how='left')

# Solo se completa con cero dentro del período efectivamente consultado en la API.
cobertura_api=df['anio'].between(2022,2026)
df.loc[cobertura_api,'n_feriados']=df.loc[cobertura_api,'n_feriados'].fillna(0)
df['nombre_aeropuerto']=df['name'].fillna(df.aeropuerto_oaci)
df['municipality']=df['municipality'].fillna('Sin información')
df['elevation_ft']=df['elevation_ft'].fillna(df.elevation_ft.median())

print('Dataset integrado:',df.shape)
print('Aeropuertos con coordenadas:',df.latitude_deg.notna().mean().round(3))
print('Cobertura de feriados: 2022 a 2026; antes de 2022 se conserva como no disponible')
df[['fecha','aeropuerto_oaci','nombre_aeropuerto','tipo_operacion','cnt_operaciones','n_feriados']].head()


Dataset integrado: (16007, 19)
Aeropuertos con coordenadas: 0.868
Cobertura de feriados: 2022 a 2026; antes de 2022 se conserva como no disponible


,fecha,aeropuerto_oaci,nombre_aeropuerto,tipo_operacion,cnt_operaciones,n_feriados
0,2000-01-01,SCAP,Alto Palena Airport,Doméstica,6,NaN
1,2000-01-01,SCAR,Chacalluta International Airport,Doméstica,691,NaN
2,2000-01-01,SCAR,Chacalluta International Airport,Internacional,139,NaN
3,2000-01-01,SCBA,Balmaceda Airport,Doméstica,382,NaN
4,2000-01-01,SCCC,Chile Chico Airport,Doméstica,82,NaN


## 4. Tres hallazgos fáciles de explicar

Cada gráfico responde una sola pregunta y su título entrega la conclusión.

In [11]:
mensual=df.groupby('fecha',as_index=False).cnt_operaciones.sum()
fig=px.line(mensual,x='fecha',y='cnt_operaciones',
    title='Hallazgo 1: la actividad cayó en 2020 y después comenzó a recuperarse',
    labels={'fecha':'','cnt_operaciones':'Operaciones mensuales'})
fig.add_vrect(x0='2020-03-01',x1='2021-01-01',fillcolor=AMARILLO,opacity=.20,line_width=0,annotation_text='Choque 2020')
fig.update_traces(line_color=AZUL,line_width=3)
fig.show()


**Interpretación:** la serie presenta estacionalidad, crecimiento de largo plazo y una ruptura excepcional durante 2020.

In [12]:
ultimos=df[df.fecha>=df.fecha.max()-pd.DateOffset(months=11)]
top=(ultimos.groupby(['aeropuerto_oaci','nombre_aeropuerto'],as_index=False).cnt_operaciones.sum()
     .nlargest(10,'cnt_operaciones').sort_values('cnt_operaciones'))
fig=px.bar(top,x='cnt_operaciones',y='aeropuerto_oaci',orientation='h',text_auto=',.0f',
    title='Hallazgo 2: Santiago concentra por amplio margen la actividad reciente',
    labels={'cnt_operaciones':'Operaciones · últimos 12 meses','aeropuerto_oaci':'Aeropuerto'})
fig.update_traces(marker_color=AZUL,textposition='outside')
fig.show()


**Interpretación:** el sistema está concentrado; por eso conviene analizar el total nacional y también cada aeropuerto.

In [13]:
# Se excluye 2026 porque contiene información solo hasta junio.
mix=df[(df.anio>=2015) & (df.anio<2026)].groupby(
    ['anio','tipo_operacion'],as_index=False
).cnt_operaciones.sum()

fig=px.bar(mix,x='anio',y='cnt_operaciones',color='tipo_operacion',barmode='stack',
    title='Hallazgo 3: las operaciones domésticas dominan el sistema (años completos 2015-2025)',
    labels={'anio':'Año','cnt_operaciones':'Operaciones','tipo_operacion':'Tipo'},
    color_discrete_sequence=[AZUL,AMARILLO])
fig.update_layout(hovermode='x unified')
fig.show()


**Interpretación:** el componente doméstico explica la mayor parte del volumen anual.

## 5. Modelo supervisado: clasificar la actividad mensual por aeropuerto

La unidad de análisis corresponde a una combinación de **aeropuerto, mes y tipo de operación**. Transformamos el problema en clasificación:

- `1`: registro con operaciones iguales o superiores a la mediana del dataset.
- `0`: registro con operaciones inferiores a la mediana.

Comparamos **regresión logística** y **árbol de decisión** mediante accuracy, precision, recall y F1. Para respetar el orden temporal, entrenamos con 2022-2025 y evaluamos con enero-junio de 2026.


In [14]:
umbral=df.cnt_operaciones.median()
df['alta_actividad']=(df.cnt_operaciones>=umbral).astype(int)
print(f'Umbral: {umbral:,.0f} operaciones mensuales por aeropuerto y tipo')
print(df.alta_actividad.value_counts(normalize=True).rename('proporción'))

variables=['anio','mes','n_feriados','es_internacional','latitude_deg',
           'longitude_deg','elevation_ft','aeropuerto_oaci']
modelo_df=df[df.anio.between(2022,2026)][variables+['alta_actividad']].dropna().copy()
X=pd.get_dummies(modelo_df[variables],columns=['aeropuerto_oaci'],drop_first=True)
y=modelo_df['alta_actividad']

# División temporal: el modelo aprende del pasado y se evalúa con 2026.
mascara_train=modelo_df['anio']<2026
X_train,X_test=X.loc[mascara_train],X.loc[~mascara_train]
y_train,y_test=y.loc[mascara_train],y.loc[~mascara_train]

# Escalado aprendido únicamente con entrenamiento para la regresión logística.
scaler_log=StandardScaler()
X_train_log=scaler_log.fit_transform(X_train)
X_test_log=scaler_log.transform(X_test)

print('Entrenamiento 2022-2025:',X_train.shape)
print('Prueba enero-junio 2026:',X_test.shape)


Umbral: 172 operaciones mensuales por aeropuerto y tipo
alta_actividad
1    0.500906
0    0.499094
Name: proporción, dtype: float64
Entrenamiento 2022-2025: (2302, 46)
Prueba enero-junio 2026: (295, 46)


In [15]:
# Modelo 1: regresión logística
logistica=LogisticRegression(max_iter=2000,random_state=RANDOM_STATE)
logistica.fit(X_train_log,y_train)
pred_log=logistica.predict(X_test_log)

# Modelo 2: árbol de decisión controlado para reducir el sobreajuste
arbol=DecisionTreeClassifier(max_depth=6,min_samples_leaf=20,random_state=RANDOM_STATE)
arbol.fit(X_train,y_train)
pred_arbol=arbol.predict(X_test)

def metricas_modelo(nombre, reales, predicciones):
    return {
        'Modelo':nombre,
        'Accuracy':accuracy_score(reales,predicciones),
        'Precision':precision_score(reales,predicciones,zero_division=0),
        'Recall':recall_score(reales,predicciones,zero_division=0),
        'F1':f1_score(reales,predicciones,zero_division=0)
    }

comparacion=pd.DataFrame([
    metricas_modelo('Regresión logística',y_test,pred_log),
    metricas_modelo('Árbol de decisión',y_test,pred_arbol)
])
comparacion.style.format({m:'{:.1%}' for m in ['Accuracy','Precision','Recall','F1']})


,Modelo,Accuracy,Precision,Recall,F1
0,Regresión logística,96.3%,93.2%,99.3%,96.1%
1,Árbol de decisión,95.3%,93.0%,97.1%,95.0%


In [16]:
comparacion_larga=comparacion.melt(
    id_vars='Modelo',var_name='Métrica',value_name='Resultado'
)
fig=px.bar(comparacion_larga,x='Métrica',y='Resultado',color='Modelo',
    barmode='group',text_auto='.1%',
    title='Comparación de modelos en datos futuros: enero-junio de 2026',
    color_discrete_sequence=[CELESTE,AZUL])
fig.update_yaxes(range=[0,1],tickformat='.0%',title='Resultado')
fig.update_layout(legend_title='Modelo')
fig.show()


In [17]:
# F1 es la métrica principal porque equilibra precision y recall de alta actividad.
mejor_fila=comparacion.sort_values(['F1','Accuracy'],ascending=False).iloc[0]
mejor_nombre=mejor_fila['Modelo']
mejor_pred=pred_arbol if mejor_nombre=='Árbol de decisión' else pred_log

cm=confusion_matrix(y_test,mejor_pred)
fig=px.imshow(cm,text_auto=True,color_continuous_scale=[[0,'#EAF6FB'],[1,AZUL]],
    x=['Predijo baja','Predijo alta'],y=['Era baja','Era alta'],
    title=f'Matriz de confusión del modelo con mayor F1: {mejor_nombre}')
fig.update_coloraxes(showscale=False)
fig.show()
print(classification_report(y_test,mejor_pred,
      target_names=['Baja actividad','Alta actividad'],zero_division=0))


                precision    recall  f1-score   support

Baja actividad       0.99      0.94      0.96       158
Alta actividad       0.93      0.99      0.96       137

      accuracy                           0.96       295
     macro avg       0.96      0.96      0.96       295
  weighted avg       0.96      0.96      0.96       295



**Cómo explicarlo:** la diagonal representa aciertos y los valores fuera de ella son errores. Accuracy mide los aciertos totales; precision indica qué proporción de las alertas de alta actividad fue correcta; recall muestra cuántos casos reales de alta actividad fueron detectados; F1 equilibra precision y recall. Se elige el modelo con mayor F1 y, en caso de empate, mayor accuracy.


## 6. Modelo no supervisado: perfiles de aeropuertos

K-Means permite descubrir grupos sin usar una etiqueta. Se escala antes de agrupar, se evalúan varios valores de K con silhouette y se utiliza PCA únicamente para visualizar.

In [18]:
perfil_base=df.groupby('aeropuerto_oaci').agg(
    operaciones_promedio=('cnt_operaciones','mean'),
    operaciones_maximas=('cnt_operaciones','max'),
    operaciones_totales=('cnt_operaciones','sum'),
    meses_activos=('fecha','nunique'),
    proporcion_internacional=('es_internacional','mean')
).reset_index()

vars_cluster=['operaciones_promedio','operaciones_maximas','operaciones_totales','meses_activos','proporcion_internacional']
scaler=StandardScaler()
X_cluster=scaler.fit_transform(perfil_base[vars_cluster])

resultados_k=[]
for k in range(2,7):
    km=KMeans(n_clusters=k,random_state=RANDOM_STATE,n_init=10)
    etiquetas=km.fit_predict(X_cluster)
    resultados_k.append({'K':k,'Inertia':km.inertia_,'Silhouette':silhouette_score(X_cluster,etiquetas)})
evaluacion_k=pd.DataFrame(resultados_k)
evaluacion_k


,K,Inertia,Silhouette
0,2,205.522635,0.735308
1,3,130.577744,0.402869
2,4,93.736451,0.422033
3,5,67.121660,0.437698
4,6,51.706423,0.465668


In [19]:
fig=px.line(evaluacion_k,x='K',y='Silhouette',markers=True,text='Silhouette',
    title='El mejor K es el que obtiene el silhouette más alto')
fig.update_traces(line_color=AZUL,texttemplate='%{text:.3f}',textposition='top center')
fig.show()


In [20]:
mejor_k=int(evaluacion_k.loc[evaluacion_k.Silhouette.idxmax(),'K'])
kmeans=KMeans(n_clusters=mejor_k,random_state=RANDOM_STATE,n_init=10)
perfil_base['cluster']=kmeans.fit_predict(X_cluster).astype(str)

pca=PCA(n_components=2)
componentes=pca.fit_transform(X_cluster)
perfil_base['PC1']=componentes[:,0]
perfil_base['PC2']=componentes[:,1]

fig=px.scatter(perfil_base,x='PC1',y='PC2',color='cluster',hover_name='aeropuerto_oaci',
    size='operaciones_totales',title=f'PCA permite visualizar los {mejor_k} perfiles de aeropuertos',
    labels={'cluster':'Cluster'})
fig.show()
print('Variabilidad representada por PCA:',pca.explained_variance_ratio_.sum().round(3))


Variabilidad representada por PCA: 0.843


In [21]:
perfil_clusters=perfil_base.groupby('cluster')[vars_cluster].mean().round(1)
perfil_clusters['cantidad_aeropuertos']=perfil_base.groupby('cluster').size()
perfil_clusters


,operaciones_promedio,operaciones_maximas,operaciones_totales,meses_activos,proporcion_internacional,cantidad_aeropuertos
cluster,,,,,,
0,378.7,1129.8,92454.1,169.8,0.1,67
1,4311.4,9147.0,2183149.5,318.0,0.3,2


**Cómo explicarlo:** los clusters no son mejores ni peores; representan perfiles distintos. PCA reduce cinco variables a dos ejes para poder dibujar los grupos.

## 7. Visualización geográfica interactiva para la demostración

Plotly permite hover, zoom, selección y descarga. Esta figura funciona como explorador geográfico interactivo; no se presenta como una aplicación de dashboard independiente. Durante la exposición se puede demostrar cómo el tamaño de cada punto representa las operaciones acumuladas.


In [22]:
mapa=(df.dropna(subset=['latitude_deg','longitude_deg'])
      .groupby(['aeropuerto_oaci','nombre_aeropuerto','latitude_deg','longitude_deg'],as_index=False)
      .cnt_operaciones.sum())
fig=px.scatter_geo(mapa,lat='latitude_deg',lon='longitude_deg',size='cnt_operaciones',
    hover_name='nombre_aeropuerto',hover_data={'aeropuerto_oaci':True,'cnt_operaciones':':,.0f','latitude_deg':False,'longitude_deg':False},
    scope='south america',projection='natural earth',
    title='Dashboard geográfico: el tamaño representa operaciones acumuladas')
fig.update_geos(fitbounds='locations',visible=True,showcountries=True,countrycolor='#B8C5D0')
fig.update_traces(marker_color=AZUL,marker_opacity=.70)
fig.show()


## 8. Dashboard interactivo con filtros

Este dashboard permite seleccionar año, aeropuerto y tipo de operación.
Los indicadores y gráficos se actualizan automáticamente.

In [26]:
from ipywidgets import interact, Dropdown
from IPython.display import display

# Habilitar widgets interactivos en Google Colab
try:
    from google.colab import output
    output.enable_custom_widget_manager()
    print("Widgets habilitados correctamente en Google Colab")
except ImportError:
    print("Entorno local: se utilizará el administrador estándar de widgets")

# Opciones de los filtros
opciones_anio = ["Todos"] + sorted(
    df["anio"].dropna().astype(int).unique().tolist()
)

opciones_aeropuerto = (
    ["Todos"] +
    sorted(df["aeropuerto_oaci"].dropna().unique().tolist())
)

opciones_tipo = (
    ["Todos"] +
    sorted(df["tipo_operacion"].dropna().unique().tolist())
)

print("Opciones del dashboard preparadas")

Widgets habilitados correctamente en Google Colab
Opciones del dashboard preparadas


In [27]:
@interact(
    anio=Dropdown(
        options=opciones_anio,
        value="Todos",
        description="Año:"
    ),
    aeropuerto=Dropdown(
        options=opciones_aeropuerto,
        value="Todos",
        description="Aeropuerto:"
    ),
    tipo=Dropdown(
        options=opciones_tipo,
        value="Todos",
        description="Tipo:"
    )
)
def dashboard_interactivo(anio, aeropuerto, tipo):

    # Copia de los datos para aplicar filtros
    filtrado = df.copy()

    if anio != "Todos":
        filtrado = filtrado[filtrado["anio"] == anio]

    if aeropuerto != "Todos":
        filtrado = filtrado[
            filtrado["aeropuerto_oaci"] == aeropuerto
        ]

    if tipo != "Todos":
        filtrado = filtrado[
            filtrado["tipo_operacion"] == tipo
        ]

    # Evitar errores si una combinación no tiene información
    if filtrado.empty:
        print("No existen datos para esta combinación de filtros.")
        return

    # Indicadores principales
    total_operaciones = filtrado["cnt_operaciones"].sum()
    promedio_mensual = filtrado["cnt_operaciones"].mean()
    cantidad_aeropuertos = filtrado["aeropuerto_oaci"].nunique()

    indicadores = pd.DataFrame({
        "Indicador": [
            "Total de operaciones",
            "Promedio por registro",
            "Aeropuertos incluidos"
        ],
        "Resultado": [
            f"{total_operaciones:,.0f}",
            f"{promedio_mensual:,.1f}",
            f"{cantidad_aeropuertos}"
        ]
    })

    display(indicadores)

    # Gráfico temporal
    serie = (
        filtrado.groupby("fecha", as_index=False)
        ["cnt_operaciones"].sum()
    )

    fig_serie = px.line(
        serie,
        x="fecha",
        y="cnt_operaciones",
        markers=True,
        title="Evolución de las operaciones según los filtros",
        labels={
            "fecha": "Fecha",
            "cnt_operaciones": "Operaciones"
        }
    )

    fig_serie.update_traces(line_color=AZUL)
    fig_serie.update_layout(hovermode="x unified")
    fig_serie.show()

    # Comparación por aeropuerto
    ranking = (
        filtrado.groupby(
            ["aeropuerto_oaci", "nombre_aeropuerto"],
            as_index=False
        )["cnt_operaciones"]
        .sum()
        .nlargest(10, "cnt_operaciones")
        .sort_values("cnt_operaciones")
    )

    fig_ranking = px.bar(
        ranking,
        x="cnt_operaciones",
        y="nombre_aeropuerto",
        orientation="h",
        text_auto=",.0f",
        title="Aeropuertos con mayor actividad",
        labels={
            "cnt_operaciones": "Operaciones",
            "nombre_aeropuerto": "Aeropuerto"
        },
        color_discrete_sequence=[CELESTE]
    )

    fig_ranking.update_layout(showlegend=False)
    fig_ranking.show()

interactive(children=(Dropdown(description='Año:', options=('Todos', 2000, 2001, 2002, 2003, 2004, 2005, 2006,…

El dashboard permite filtrar por año, aeropuerto y tipo de operación. Cada selección actualiza los indicadores, la evolución temporal y el ranking. Esto permite que un usuario no técnico explore los datos sin modificar el código

## 9. Conclusiones y recomendaciones

1. La actividad aérea chilena muestra estacionalidad, crecimiento y una ruptura extraordinaria en 2020.
2. El sistema presenta concentración en los aeropuertos con mayor volumen, especialmente Santiago.
3. La clasificación estima actividad alta o baja para cada combinación mensual de aeropuerto y tipo de operación.
4. La evaluación temporal con 2026 es más exigente y realista que mezclar aleatoriamente registros del pasado y del futuro.
5. K-Means identifica perfiles estadísticos diferentes de aeropuertos y PCA permite visualizarlos en dos dimensiones.
6. La visualización geográfica comunica los resultados de manera interactiva para usuarios no técnicos.

### Recomendación

Actualizar mensualmente las fuentes y utilizar el modelo como señal de apoyo para planificación, complementándolo con eventos especiales, capacidad aeroportuaria y cambios operacionales.


## 9. Limitaciones responsables

- El umbral de alta actividad depende de la mediana y puede ajustarse según la decisión operacional.
- La unidad analizada es aeropuerto-mes-tipo de operación; no representa el total nacional de un mes.
- Una asociación encontrada por el modelo no demuestra causalidad.
- La API de feriados cubre 2022-2026; los años anteriores se conservan como información no disponible.
- Los feriados nacionales no incluyen todos los eventos regionales.
- Los datos de 2026 llegan solamente hasta junio y no deben compararse como año completo.
- El modelo debe reentrenarse cuando se incorporen nuevos meses.
- Los clusters describen similitudes estadísticas, no categorías oficiales.


## 10. Evidencias y archivos de entrega

### Evidencias presentes en este notebook

- **Código:** carga, validación, ETL, integración, modelos y visualizaciones organizados por etapas.
- **Datos:** referencias a las tres fuentes y controles de calidad del CSV principal.
- **Modelos:** comparación supervisada, matriz de confusión, K-Means, silhouette y PCA.
- **Documentación:** metodología, resultados, decisiones y limitaciones.

### Evidencias que solo deben declararse si se adjuntan realmente

- Repositorio Git/GitHub con historial de commits.
- `Dockerfile` y `requirements.txt` probados.
- Configuración de CI/CD y evidencia de una ejecución exitosa.

### Fuentes

- Junta de Aeronáutica Civil / Datos.gob.cl: https://datos.gob.cl/dataset/operaciones-aeronaves
- Nager.Date API: https://date.nager.at/Api
- OurAirports: https://ourairports.com/data/

### Cierre para la exposición

> “Integramos tres fuentes, validamos y transformamos los datos, comparamos dos modelos supervisados con una evaluación temporal, segmentamos aeropuertos con K-Means y comunicamos los resultados mediante visualizaciones interactivas.”
